In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
import torchvision.utils as vutils
import numpy as np
import os
from medmnist import PathMNIST
from tqdm import tqdm
import torch.nn.functional as F
from torchvision.models import inception_v3
from scipy import linalg

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
num_epochs = 50
batch_size = 128
lr_d = 0.0001
lr_g = 0.0004
z_dim = 100
n_critic = 5  # Number of critic updates per generator update
clip_value = 0.01  # Weight clipping range for decent results

# Data loading and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = PathMNIST(split='train', transform=transform, download=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,  # Multi-threading for faster data loading
    pin_memory=True  # Faster GPU transfer
)

Using device: cuda
Using downloaded and verified file: C:\Users\Sudhanshu\.medmnist\pathmnist.npz


In [3]:
# Generator for WGAN
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 2048, 4, 1, 0, bias=False),
            nn.BatchNorm2d(2048),
            nn.ReLU(True),
            nn.ConvTranspose2d(2048, 1024, 4, 2, 1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(True),
            nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# Critic for WGAN with Weight Clipping
class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 1, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.model(x)

In [4]:
# Evaluation functions
def load_inception_model(device):
    inception_model = inception_v3(weights='Inception_V3_Weights.IMAGENET1K_V1').to(device)
    inception_model.eval()
    return inception_model

def get_inception_activations(images, inception_model, device, batch_size=32):
    images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    activations = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i + batch_size].to(device)
            act = inception_model(batch)
            activations.append(act.cpu().numpy())
    return np.concatenate(activations, axis=0)

def compute_fid(real_images, fake_images, inception_model, device, batch_size=32):
    real_acts = get_inception_activations(real_images, inception_model, device, batch_size)
    fake_acts = get_inception_activations(fake_images, inception_model, device, batch_size)
    mu_real, sigma_real = np.mean(real_acts, axis=0), np.cov(real_acts, rowvar=False)
    mu_fake, sigma_fake = np.mean(fake_acts, axis=0), np.cov(fake_acts, rowvar=False)
    diff = mu_real - mu_fake
    covmean = linalg.sqrtm(sigma_real.dot(sigma_fake), disp=False)[0]
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def compute_inception_score(images, inception_model, device, splits=10, batch_size=32):
    images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    preds = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i + batch_size].to(device)
            pred = inception_model(batch)
            pred = F.softmax(pred, dim=1).cpu().numpy()
            preds.append(pred)
    preds = np.concatenate(preds, axis=0)
    scores = []
    for i in range(splits):
        part = preds[(i * preds.shape[0] // splits):((i + 1) * preds.shape[0] // splits)]
        kl = part * (np.log(part) - np.log(np.mean(part, axis=0, keepdims=True)))
        kl = np.mean(np.sum(kl, axis=1))
        scores.append(np.exp(kl))
    return np.mean(scores), np.std(scores)

def evaluate(generator, train_loader, device, num_samples=5000, z_dim=100, batch_size=128):
    inception_model = load_inception_model(device)
    generator.eval()
    fake_images = []
    with torch.no_grad():
        for _ in range(num_samples // batch_size):
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake = generator(z)
            fake_images.append(fake.cpu())
    fake_images = torch.cat(fake_images, dim=0)[:num_samples]
    
    real_images = []
    for batch in train_loader:
        real = batch[0]
        real_images.append(real)
        if len(real_images) * batch_size >= num_samples:
            break
    real_images = torch.cat(real_images, dim=0)[:num_samples]
    
    fid_score = compute_fid(real_images, fake_images, inception_model, device, batch_size)
    is_mean, is_std = compute_inception_score(fake_images, inception_model, device, batch_size)
    return fid_score, is_mean, is_std

In [5]:
# Training function for WGAN with Weight Clipping
def train_wgan():
    # Initialize models
    generator = Generator(z_dim=z_dim).to(device)
    critic = Critic().to(device)
    
    # Optimizers
    optimizer_c = optim.Adam(critic.parameters(), lr=lr_d, betas=(0.5, 0.999))
    optimizer_g = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.5, 0.999))
    
    # TensorBoard writer
    writer = SummaryWriter('runs/wgan_pathmnist')
    
    # Create directory for saving images
    os.makedirs('generated_images_WGAN', exist_ok=True)
    
    # Best model tracking
    best_c_loss = float('inf')
    
    # Fixed noise for visualization
    fixed_z = torch.randn(5, z_dim, 1, 1).to(device)
    
    # Training loop
    for epoch in range(num_epochs):
        critic.train()
        generator.train()
        c_loss_total = 0.0
        g_loss_total = 0.0
        
        for i, (real_images, _) in enumerate(tqdm(train_loader, desc=f"WGAN Epoch {epoch+1}/{num_epochs}")):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            # Train Critic
            for _ in range(n_critic):
                optimizer_c.zero_grad()
                real_output = critic(real_images)
                c_loss_real = -real_output.mean()
                
                z = torch.randn(batch_size, z_dim, 1, 1).to(device)
                fake_images = generator(z)
                fake_output = critic(fake_images.detach())
                c_loss_fake = fake_output.mean()
                
                c_loss = c_loss_real + c_loss_fake
                c_loss.backward()
                optimizer_c.step()
                
                # Weight clipping
                for p in critic.parameters():
                    p.data.clamp_(-clip_value, clip_value)
            
            c_loss_total += c_loss.item()
            
            # Train Generator
            optimizer_g.zero_grad()
            fake_output = critic(fake_images)
            g_loss = -fake_output.mean()
            g_loss.backward()
            optimizer_g.step()
            
            g_loss_total += g_loss.item()
            
            # Compute c_wasserstein for logging
            c_wasserstein = (real_output.mean() - fake_output.mean()).item()
        
        # Average losses for the epoch
        c_loss_avg = c_loss_total / len(train_loader)
        g_loss_avg = g_loss_total / len(train_loader)
        
        # Log to TensorBoard
        writer.add_scalar('Loss/Critic', c_loss_avg, epoch)
        writer.add_scalar('Loss/Generator', g_loss_avg, epoch)
        writer.add_scalar('Metrics/C_Wasserstein', c_wasserstein, epoch)
        
        print(f"WGAN Epoch {epoch+1}: c_loss={c_loss_avg:.4f}, g_loss={g_loss_avg:.4f}, c_wasserstein={c_wasserstein:.4f}")
        
        # Save best model based on c_loss
        if c_loss_avg < best_c_loss:
            best_c_loss = c_loss_avg
            torch.save(generator.state_dict(), 'best_generator_wgan.pth')
            print(f"Saved best WGAN generator at epoch {epoch+1}")
        
        # Visualize every 5 epochs with TensorBoard
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                fake_images = generator(fixed_z)
                real_images_vis = (real_images[:5] + 1) / 2  # Denormalize
                fake_images_vis = (fake_images + 1) / 2  # Denormalize
                real_grid = vutils.make_grid(real_images_vis, nrow=5, padding=2, normalize=False)
                fake_grid = vutils.make_grid(fake_images_vis, nrow=5, padding=2, normalize=False)
                writer.add_image('Images/Real', real_grid, epoch)
                writer.add_image('Images/Generated', fake_grid, epoch)
                # Save to disk as well
                vutils.save_image(fake_images_vis, f"generated_images_WGAN/epoch_{epoch+1}.png", nrow=5, normalize=False)
    
    # Final evaluation after training
    print("Training completed. Performing final evaluation...")
    fid_score, is_mean, is_std = evaluate(generator, train_loader, device, num_samples=5000, z_dim=z_dim)
    print(f"Final FID Score: {fid_score:.2f}")
    print(f"Final Inception Score: {is_mean:.2f} ± {is_std:.2f}")
    
    # Log final metrics to TensorBoard
    writer.add_scalar('Metrics/Final FID', fid_score, num_epochs)
    writer.add_scalar('Metrics/Final Inception Score Mean', is_mean, num_epochs)
    writer.add_scalar('Metrics/Final Inception Score Std', is_std, num_epochs)
    writer.close()

In [6]:
# Run the training
if __name__ == "__main__":
    train_wgan()

WGAN Epoch 1/50: 100%|██████████| 704/704 [07:20<00:00,  1.60it/s]


WGAN Epoch 1: c_loss=-818.2505, g_loss=-55.1663, c_wasserstein=805.8696
Saved best WGAN generator at epoch 1


WGAN Epoch 2/50: 100%|██████████| 704/704 [07:27<00:00,  1.57it/s]


WGAN Epoch 2: c_loss=-832.0366, g_loss=-62.5489, c_wasserstein=805.8397
Saved best WGAN generator at epoch 2


WGAN Epoch 3/50: 100%|██████████| 704/704 [07:25<00:00,  1.58it/s]


...

WGAN Epoch 49/50: 100%|██████████| 704/704 [07:49<00:00,  1.50it/s]


WGAN Epoch 49: c_loss=-830.1222, g_loss=-62.0706, c_wasserstein=640.2364


WGAN Epoch 50/50: 100%|██████████| 704/704 [07:52<00:00,  1.49it/s]


WGAN Epoch 50: c_loss=-830.6664, g_loss=-61.7021, c_wasserstein=1050.3771
Training completed. Performing final evaluation...


RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 5355477504 bytes.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PathMNIST
from torch.utils.tensorboard import SummaryWriter

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
z_dim = 100
batch_size = 32
num_samples = 500  # Reduced to fit memory

# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = PathMNIST(split='train', transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

# Generator class (must match your trained model)
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 2048, 4, 1, 0, bias=False),
            nn.BatchNorm2d(2048),
            nn.ReLU(True),
            nn.ConvTranspose2d(2048, 1024, 4, 2, 1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(True),
            nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# Load the trained generator
generator = Generator(z_dim=z_dim).to(device)
generator.load_state_dict(torch.load('best_generator_wgan.pth'))
generator.eval()

# Generate fake images and add to tensorboard.
fake_images = []

with torch.no_grad():
    for _ in range(num_samples // batch_size):
        z = torch.randn(batch_size, z_dim, 1, 1).to(device)
        fake = generator(z)
        fake_images.append(fake.cpu())
fake_images = torch.cat(fake_images, dim=0)[:num_samples]

# Set up TensorBoard writer
writer = SummaryWriter('runs/wgan_generator_samples')

# Add images to TensorBoard
writer.add_images('Generated Images', (fake_images + 1) / 2, 0) # normalize to 0-1 range.

#Add the model graph to tensorboard
z = torch.randn(1, z_dim, 1,1).to(device)
writer.add_graph(generator, z)

print("TensorBoard is ready. Run 'tensorboard --logdir=runs' in your terminal.")
writer.close()

Using device: cuda
Using downloaded and verified file: C:\Users\Sudhanshu\.medmnist\pathmnist.npz


C:\Users\Sudhanshu\AppData\Local\Temp\ipykernel_29576\1703928768.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generator.load_state_dict(torch.load('best_generator_wg

TensorBoard is ready. Run 'tensorboard --logdir=runs' in your terminal.
